# Corte 3, Punto 1: deduplicación y capas silver y gold

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T1.1 a T1.6 (diagnóstico de duplicados, silver, gold, tablas resumen, reconciliación y linaje).

Entrada: bronze. Salidas: `silver/`, `gold/` y `procesado/`.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T1.1 Diagnóstico de duplicados y elección de la clave natural

El bronze conserva todas las filas válidas del CSV, pero un mismo proceso de contratación puede aparecer varias veces (nuevas publicaciones, lotes o varias adjudicaciones). Antes de deduplicar hay que medir cuánto se repite cada clave candidata y entender por qué, para elegir la clave que define **un contrato adjudicado**.

Claves candidatas:
- **Fila completa:** filas idénticas en todas las columnas (duplicado exacto).
- **`id_del_proceso`:** un registro por proceso.
- **`id_del_proceso` + `id_adjudicacion`:** un registro por adjudicación dentro de un proceso.
- **`id_adjudicacion`:** identificador de la adjudicación por sí solo.

Se mide sobre todo el bronze y sobre los registros con `adjudicado = True`, que son los que pasarán a gold.

In [2]:
# T1.1 · Bronze de entrada
bronze = spark.read.parquet(BRONZE_PARQUET_UNPARTITIONED)

# fecha_ingesta_utc la pone el proceso de carga; no describe el contrato
columnas_contenido = [c for c in bronze.columns if c != "fecha_ingesta_utc"]

filas_bronze = bronze.count()
filas_adjudicadas = bronze.filter(F.col("adjudicado") == True).count()
print(f"Filas en el bronze:      {filas_bronze:,}")
print(f"Filas con adjudicado=True: {filas_adjudicadas:,}")
print(f"Columnas de contenido:   {len(columnas_contenido)}")

Filas en el bronze:      9,099,432
Filas con adjudicado=True: 872,756
Columnas de contenido:   59


### Tabla de diagnóstico por clave candidata

*Filas sobrantes* = filas − claves distintas: son las filas que desaparecerían si se dejara una sola por clave.

In [3]:
# T1.1 · Tabla de diagnóstico: filas, claves distintas y filas sobrantes por clave candidata
from functools import reduce
from operator import or_
import pandas as pd

claves_candidatas = {
    "fila completa (duplicado exacto)": columnas_contenido,
    "id_del_proceso": ["id_del_proceso"],
    "id_del_proceso + id_adjudicacion": ["id_del_proceso", "id_adjudicacion"],
    "id_adjudicacion": ["id_adjudicacion"],
}

def diagnosticar(df, universo):
    filas = df.count()
    resultado = []
    for nombre, columnas in claves_candidatas.items():
        if len(columnas) > 2:
            # Fila completa: se compara un hash de 64 bits de todas las columnas
            # (con 9 millones de filas la probabilidad de colisión es despreciable)
            distintas = df.select(F.xxhash64(*columnas).alias("h")).distinct().count()
            nulas = "n/a"
        else:
            distintas = df.select(*columnas).distinct().count()
            nulas = df.filter(reduce(or_, [F.col(c).isNull() for c in columnas])).count()
        resultado.append({
            "universo": universo,
            "clave candidata": nombre,
            "filas": filas,
            "claves distintas": distintas,
            "filas sobrantes": filas - distintas,
            "% sobrante": round(100 * (filas - distintas) / filas, 2) if filas else 0.0,
            "filas con clave nula": nulas,
        })
    return resultado

diagnostico_claves = pd.DataFrame(
    diagnosticar(bronze, "todo el bronze")
    + diagnosticar(bronze.filter(F.col("adjudicado") == True), "solo adjudicados")
)
display(diagnostico_claves)

,universo,clave candidata,filas,claves distintas,filas sobrantes,% sobrante,filas con clave nula
0,todo el bronze,fila completa (duplicado exacto),9099432,8963967,135465,1.49,n/a
1,todo el bronze,id_del_proceso,9099432,8667332,432100,4.75,0
2,todo el bronze,id_del_proceso + id_adjudicacion,9099432,8694126,405306,4.45,8226676
3,todo el bronze,id_adjudicacion,9099432,547419,8552013,93.98,8226676
4,solo adjudicados,fila completa (duplicado exacto),872756,817259,55497,6.36,n/a
5,solo adjudicados,id_del_proceso,872756,520624,352132,40.35,0
6,solo adjudicados,id_del_proceso + id_adjudicacion,872756,547418,325338,37.28,0
7,solo adjudicados,id_adjudicacion,872756,547418,325338,37.28,0


### Distribución de repeticiones por proceso

In [4]:
# T1.1 · ¿Cuántas veces se repite cada id_del_proceso?
repeticiones_proceso = (
    bronze.groupBy("id_del_proceso").count()
    .withColumn("veces", F.least(F.col("count"), F.lit(10)))
    .groupBy("veces")
    .agg(F.count("*").alias("procesos"), F.sum("count").alias("filas"))
    .orderBy("veces")
    .collect()
)
repeticiones_proceso = pd.DataFrame([r.asDict() for r in repeticiones_proceso])
repeticiones_proceso["veces"] = repeticiones_proceso["veces"].map(lambda v: "10 o más" if v == 10 else str(v))
display(repeticiones_proceso)

,veces,procesos,filas
0,1,8583881,8583881
1,2,50766,101532
2,3,10958,32874
3,4,8696,34784
4,5,2734,13670
5,6,2271,13626
6,7,761,5327
7,8,1041,8328
8,9,1310,11790
9,10 o más,4914,293620


### Qué cambia entre las filas de un mismo proceso

Si en un proceso repetido cambia `id_adjudicacion`, las filas son adjudicaciones distintas (lotes o varios contratistas). Si solo cambian fechas de publicación o el estado, son versiones sucesivas del mismo proceso.

In [5]:
# T1.1 · Dentro de un mismo id_del_proceso repetido, ¿qué columnas cambian?
# Para cada proceso repetido se cuenta cuántos valores distintos tiene cada columna
# (se usa un hash por valor para no cargar textos largos en memoria; el nulo cuenta como valor).
procesos_repetidos = (
    bronze.groupBy("id_del_proceso").count()
    .filter(F.col("count") > 1)
    .select("id_del_proceso")
)
columnas_a_revisar = [c for c in columnas_contenido if c != "id_del_proceso"]

variacion = (
    bronze.join(procesos_repetidos, "id_del_proceso", "left_semi")
    .groupBy("id_del_proceso")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_a_revisar])
)
total_repetidos = procesos_repetidos.count()

fila = variacion.agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_a_revisar]).first()
columnas_que_cambian = pd.DataFrame(
    [(c, fila[c] or 0, round(100 * (fila[c] or 0) / total_repetidos, 2) if total_repetidos else 0.0)
     for c in columnas_a_revisar],
    columns=["columna", "procesos repetidos donde cambia", "% de procesos repetidos"],
).sort_values("procesos repetidos donde cambia", ascending=False)

print(f"Procesos con más de una fila: {total_repetidos:,}")
display(columnas_que_cambian[columnas_que_cambian["procesos repetidos donde cambia"] > 0].reset_index(drop=True))

Procesos con más de una fila: 83,451


,columna,procesos repetidos donde cambia,% de procesos repetidos
0,valor_total_adjudicacion,22389,26.83
1,id_adjudicacion,18308,21.94
2,nombre_proveedor_adjudicado,14393,17.25
3,codigo_proveedor,14393,17.25
4,fecha_adjudicacion,14099,16.89
5,nit_proveedor_adjudicado,10433,12.50
6,departamento_proveedor,9889,11.85
7,ciudad_proveedor,9541,11.43
8,nombre_adjudicador,2582,3.09


### Muestra de procesos repetidos para revisión manual

In [6]:
# T1.1 · Muestra reproducible de procesos repetidos para revisarlos a mano
# Se eligen los 5 procesos repetidos con el hash más bajo: siempre salen los mismos.
columnas_muestra = [
    "id_del_proceso", "referencia_del_proceso", "fase", "estado_procedimiento",
    "adjudicado", "id_adjudicacion", "numero_lotes",
    "fecha_publicacion_proceso", "fecha_ultima_publicacion", "fecha_adjudicacion",
    "precio_base", "valor_total_adjudicacion",
    "nit_proveedor_adjudicado", "nombre_proveedor_adjudicado",
]
ids_muestra = [
    r.id_del_proceso
    for r in procesos_repetidos.orderBy(F.xxhash64("id_del_proceso"), "id_del_proceso").limit(5).collect()
]
muestra_repetidos = (
    bronze.filter(F.col("id_del_proceso").isin(ids_muestra))
    .select(*columnas_muestra)
    .orderBy("id_del_proceso", "fecha_ultima_publicacion", "id_adjudicacion")
    .collect()
)
muestra_repetidos = pd.DataFrame([r.asDict() for r in muestra_repetidos], columns=columnas_muestra)
pd.set_option("display.max_columns", None)
display(muestra_repetidos)

,id_del_proceso,referencia_del_proceso,fase,estado_procedimiento,adjudicado,id_adjudicacion,numero_lotes,fecha_publicacion_proceso,fecha_ultima_publicacion,fecha_adjudicacion,precio_base,valor_total_adjudicacion,nit_proveedor_adjudicado,nombre_proveedor_adjudicado
0,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.300105,0,2018-04-10,2018-04-10,NaT,103805319.00,101533300.00,None,VICARTECHZ SAS
1,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.301134,0,2018-04-10,2018-04-10,2018-06-08,103805319.00,95532681.00,None,TEAM CORP SAS
2,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
3,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
4,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
5,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
6,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
7,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
8,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
9,CO1.REQ.6296380,413-MC-026-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1941961,0,2024-05-27,2024-05-27,NaT,5130000.00,4288811.00,None,SERHE H2O


In [7]:
# T1.1 · Dentro de una misma adjudicación, ¿qué cambia cuando las filas NO son duplicados exactos?
adjudicados_sin_exactos = (
    bronze.filter(F.col("adjudicado") == True)
    .dropDuplicates(columnas_contenido)
)
adjudicaciones_repetidas = (
    adjudicados_sin_exactos.groupBy("id_adjudicacion").count()
    .filter(F.col("count") > 1)
    .select("id_adjudicacion")
)
filas_adj_sin_exactos = adjudicados_sin_exactos.count()
total_adj_repetidas = adjudicaciones_repetidas.count()

columnas_adj = [c for c in columnas_contenido if c != "id_adjudicacion"]
fila_adj = (
    adjudicados_sin_exactos.join(adjudicaciones_repetidas, "id_adjudicacion", "left_semi")
    .groupBy("id_adjudicacion")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_adj])
    .agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_adj])
    .first()
)
cambios_adjudicacion = pd.DataFrame(
    [(c, fila_adj[c] or 0, round(100 * (fila_adj[c] or 0) / total_adj_repetidas, 2) if total_adj_repetidas else 0.0)
     for c in columnas_adj],
    columns=["columna", "adjudicaciones donde cambia", "% de adjudicaciones repetidas"],
).sort_values("adjudicaciones donde cambia", ascending=False)

print(f"Filas adjudicadas sin duplicados exactos: {filas_adj_sin_exactos:,}")
print(f"Adjudicaciones distintas:                 {adjudicados_sin_exactos.select('id_adjudicacion').distinct().count():,}")
print(f"Adjudicaciones con más de una versión:    {total_adj_repetidas:,}")
display(cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0].reset_index(drop=True))

Filas adjudicadas sin duplicados exactos: 817,259
Adjudicaciones distintas:                 547,418
Adjudicaciones con más de una versión:    16,616


,columna,adjudicaciones donde cambia,% de adjudicaciones repetidas
0,valor_total_adjudicacion,16110,96.95
1,codigo_proveedor,10571,63.62
2,nombre_proveedor_adjudicado,10571,63.62
3,nit_proveedor_adjudicado,7893,47.50
4,departamento_proveedor,7333,44.13
5,ciudad_proveedor,7094,42.69
6,fecha_adjudicacion,163,0.98


In [8]:
# T1.1 · Muestra reproducible de adjudicaciones con varias versiones distintas
columnas_que_varian = cambios_adjudicacion.loc[
    cambios_adjudicacion["adjudicaciones donde cambia"] > 0, "columna"
].tolist()[:10]
ids_adj_muestra = [
    r.id_adjudicacion
    for r in adjudicaciones_repetidas.orderBy(F.xxhash64("id_adjudicacion"), "id_adjudicacion").limit(5).collect()
]
muestra_adjudicaciones = (
    adjudicados_sin_exactos.filter(F.col("id_adjudicacion").isin(ids_adj_muestra))
    .select("id_adjudicacion", "id_del_proceso", *columnas_que_varian)
    .orderBy("id_adjudicacion", *columnas_que_varian)
    .collect()
)
display(pd.DataFrame([r.asDict() for r in muestra_adjudicaciones],
                     columns=["id_adjudicacion", "id_del_proceso", *columnas_que_varian]))

,id_adjudicacion,id_del_proceso,valor_total_adjudicacion,codigo_proveedor,nombre_proveedor_adjudicado,nit_proveedor_adjudicado,departamento_proveedor,ciudad_proveedor,fecha_adjudicacion
0,CO1.AWD.1346458,CO1.REQ.3045146,50000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
1,CO1.AWD.1346458,CO1.REQ.3045146,55000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
2,CO1.AWD.1346458,CO1.REQ.3045146,75000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
3,CO1.AWD.1492664,CO1.REQ.3662832,106200000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
4,CO1.AWD.1492664,CO1.REQ.3662832,213800000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
5,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
6,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
7,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
8,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
9,CO1.AWD.2315486,CO1.REQ.8427888,1606596857.00,732170105,CONSORCIO VIRREY ARC,901973576,None,Funza,2025-07-28


### Decisión sobre la clave natural

In [9]:
# T1.1 · Decisión sobre la clave natural (las cifras salen de las celdas anteriores)
def _fila(universo, clave):
    return diagnostico_claves[
        (diagnostico_claves["universo"] == universo) & (diagnostico_claves["clave candidata"] == clave)
    ].iloc[0]

clave_adj = _fila("solo adjudicados", "id_del_proceso + id_adjudicacion")
solo_awd = _fila("solo adjudicados", "id_adjudicacion")
exactos_adj = filas_adjudicadas - filas_adj_sin_exactos
sobrantes_conflicto = filas_adj_sin_exactos - int(clave_adj["claves distintas"])
cambian = cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0]
lista_cambian = ", ".join(f"`{c}` ({n:,})" for c, n in zip(cambian["columna"], cambian["adjudicaciones donde cambia"]))

display(Markdown(f"""
**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas {filas_adjudicadas:,} filas.
- Hay **{int(clave_adj['claves distintas']):,}** parejas proceso + adjudicación distintas y **{int(solo_awd['claves distintas']):,}** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **{int(_fila('solo adjudicados', 'id_del_proceso')['claves distintas']):,}** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las {filas_adjudicadas - int(clave_adj['claves distintas']):,} filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** {exactos_adj:,} filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** {sobrantes_conflicto:,} filas en {total_adj_repetidas:,} adjudicaciones ({100 * total_adj_repetidas / int(clave_adj['claves distintas']):.2f} % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: {lista_cambian}. El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.
"""))


**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas 872,756 filas.
- Hay **547,418** parejas proceso + adjudicación distintas y **547,418** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **520,624** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las 325,338 filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** 55,497 filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** 269,841 filas en 16,616 adjudicaciones (3.04 % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: `valor_total_adjudicacion` (16,110), `codigo_proveedor` (10,571), `nombre_proveedor_adjudicado` (10,571), `nit_proveedor_adjudicado` (7,893), `departamento_proveedor` (7,333), `ciudad_proveedor` (7,094), `fecha_adjudicacion` (163). El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.
